# NOVA RETAIL — Reconciliación de SKUs

## Limpieza y homologación entre SAP y AS400
**Proyecto:** Diagnóstico de Prevención de Pérdidas  
**Autor:** Denz One  
**Fase:** Limpieza de datos y reconciliación de catálogos  
**Objetivo:** Construir una estrategia de homologación entre los catálogos SAP y AS400 para identificar equivalencias, detectar productos invisibles y preparar la base para la reconciliación de inventario.

---

### Propósito de este notebook
Este notebook tiene como objetivo resolver uno de los problemas más críticos del caso NOVA RETAIL:

- SAP y AS400 no comparten una tabla maestra de equivalencias
- un mismo producto puede existir con descripciones degradadas o formatos distintos
- ciertos productos de alto valor no existen en AS400
- sin reconciliar SKUs, cualquier análisis de merma o fraude queda contaminado

### Preguntas clave
1. ¿Qué tan comparable es el catálogo SAP con el catálogo AS400?
2. ¿Cuántos productos pueden reconciliarse automáticamente?
3. ¿Cuáles requieren revisión manual?
4. ¿Cuáles son invisibles para AS400 y representan riesgo estructural?

### Nota del analista
La reconciliación de SKUs no es un ejercicio cosmético de limpieza.  
Es una condición necesaria para poder estimar pérdida real, separar ruido operativo y detectar vulnerabilidades explotables en la cadena de inventario.

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

DATA_PATH = Path("../data/raw")

products_sap = pd.read_csv(DATA_PATH / "products_sap.csv")
products_as400 = pd.read_csv(DATA_PATH / "products_as400.csv")

print("Datasets cargados correctamente.")
print("SAP:", products_sap.shape)
print("AS400:", products_as400.shape)

Datasets cargados correctamente.
SAP: (14000, 6)
AS400: (14000, 6)


In [6]:
products_sap.head()

,sku_sap,description_sap,category_sap,brand,price_sap,supplier_id
0,SAP-0000001,"PANTALLA SAMSUNG 75""",ELECTRONICA,SAMSUNG,10081.43,PROV-009
1,SAP-0000002,"BOCINA TCL 32""",ELECTRONICA,TCL,20521.09,PROV-046
2,SAP-0000003,MONITOR JBL 256GB,ELECTRONICA,JBL,13228.27,PROV-010
3,SAP-0000004,"CAMARA TCL 75""",ELECTRONICA,TCL,28635.23,PROV-047
4,SAP-0000005,"CAMARA TCL 65""",ELECTRONICA,TCL,25250.87,PROV-003


In [7]:
summary = pd.DataFrame({
    "dataset": ["products_sap", "products_as400"],
    "filas": [products_sap.shape[0], products_as400.shape[0]],
    "columnas": [products_sap.shape[1], products_as400.shape[1]]
})

summary

,dataset,filas,columnas
0,products_sap,14000,6
1,products_as400,14000,6


In [8]:
apple_ghosts = products_as400[products_as400["is_ghost"] == True]

print(f"Total Apple Ghosts: {len(apple_ghosts)}")
print("\nPrimeros 5 Apple Ghosts:")
apple_ghosts.head()

Total Apple Ghosts: 22

Primeros 5 Apple Ghosts:


,sku_as400,description_as400,category_as400,price_as400,sku_sap_reference,is_ghost
2102,NaN,NaN,NaN,NaN,SAP-0002103,True
2112,NaN,NaN,NaN,NaN,SAP-0002113,True
2114,NaN,NaN,NaN,NaN,SAP-0002115,True
2119,NaN,NaN,NaN,NaN,SAP-0002120,True
2126,NaN,NaN,NaN,NaN,SAP-0002127,True


In [9]:
as400_quality = {
    "total_registros": len(products_as400),
    "sku_as400_nulos": products_as400["sku_as400"].isna().sum(),
    "description_as400_nulos": products_as400["description_as400"].isna().sum(),
    "category_as400_vacias": (products_as400["category_as400"] == "").sum(),
    "price_as400_nulos": products_as400["price_as400"].isna().sum(),
    "apple_ghosts": products_as400["is_ghost"].sum()
}

pd.DataFrame(as400_quality.items(), columns=["metrica", "valor"])

,metrica,valor
0,total_registros,14000
1,sku_as400_nulos,22
2,description_as400_nulos,22
3,category_as400_vacias,0
4,price_as400_nulos,589
5,apple_ghosts,22


In [10]:
corrupt_examples = products_as400[
    products_as400["description_as400"].notna() &
    products_as400["description_as400"].str.contains("SAMSNG|APLE|TV|PANT|REFRI|LICUAD|0|3", na=False)
].head(10)

corrupt_examples[["sku_as400", "description_as400", "price_as400"]]

,sku_as400,description_as400,price_as400
0,ELEC-9935,PANTALLA S,582.74
1,00000002,"BOCINA TCL 32""",20521.09
7,"TELEVISOR LG 32""","TELEVISOR LG 32""",28566.51
11,"MONITOR HISENSE 32""","MONITOR HISENSE 32""",33475.93
13,ELEC-2403,"AUDIFONOS LG 32""",9948.76
16,603-BOCI-29,BOCINA SAMSNG 128GB,2694.13
17,553-PANT-17,"PANTALLA TCL 65""",11189.75
20,ELEC-1828,"TABLET LG 43""",16623.31
21,886-PANT-52,"PANTALLA LG 55""",15411.66
24,414-PANT-16,"PANTALLA LG 43""",21636.16


In [11]:
price_nulls_by_category = (
    products_as400[products_as400["price_as400"].isna()]
    ["category_as400"]
    .value_counts()
    .head(10)
)

price_nulls_by_category

category_as400
ALIMENTOS            81
ELECTRONICA          61
ELECTRODOMESTICOS    56
LIMPIEZA             56
COMPUTO              44
TELEFONIA            33
CUIDADO_PERSONAL     32
LICORES              32
MISCELANEOS          15
ALIM                  7
Name: count, dtype: int64

In [12]:
import plotly.express as px

quality_metrics = pd.DataFrame({
    "metrica": ["SKU nulos", "Descripciones nulas", "Categorías vacías", "Precios nulos", "Apple Ghosts"],
    "cantidad": [
        products_as400["sku_as400"].isna().sum(),
        products_as400["description_as400"].isna().sum(),
        (products_as400["category_as400"] == "").sum(),
        products_as400["price_as400"].isna().sum(),
        products_as400["is_ghost"].sum()
    ]
})

fig = px.bar(
    quality_metrics,
    x="metrica",
    y="cantidad",
    title="Problemas de calidad en catálogo AS400",
    color="cantidad",
    color_continuous_scale="Reds"
)

fig.show()

In [13]:
import re

def normalize_text(text):
    if pd.isna(text):
        return None

    text = str(text).upper().strip()

    replacements = {
        "SAMSNG": "SAMSUNG",
        "APLE": "APPLE",
        "TV": "TELEVISOR",
        "PANT": "PANTALLA",
        "REFRI": "REFRIGERADOR",
        "LICUAD": "LICUADORA",
        "0": "O",
        "3": "E",
        "_": " "
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(r'[^A-Z0-9" ]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [14]:
products_sap["desc_norm"] = products_sap["description_sap"].apply(normalize_text)
products_as400["desc_norm"] = products_as400["description_as400"].apply(normalize_text)

products_sap[["description_sap", "desc_norm"]].head()

,description_sap,desc_norm
0,"PANTALLA SAMSUNG 75""","PANTALLAALLA SAMSUNG 75"""
1,"BOCINA TCL 32""","BOCINA TCL E2"""
2,MONITOR JBL 256GB,MONITOR JBL 256GB
3,"CAMARA TCL 75""","CAMARA TCL 75"""
4,"CAMARA TCL 65""","CAMARA TCL 65"""


In [15]:
products_as400[["description_as400", "desc_norm"]].head(15)

,description_as400,desc_norm
0,PANTALLA S,PANTALLAALLA S
1,"BOCINA TCL 32""","BOCINA TCL E2"""
2,MONITOR JBL 256GB,MONITOR JBL 256GB
3,"CAMARA TCL 75""","CAMARA TCL 75"""
4,"CAMARA TCL 65""","CAMARA TCL 65"""
5,TABLET JBL 128GB,TABLET JBL 128GB
6,"MONITOR TCL 65""","MONITOR TCL 65"""
7,"TELEVISOR LG 32""","TELEVISOR LG E2"""
8,"TABLET LG 75""","TABLET LG 75"""
9,MONITOR TCL 512GB,MONITOR TCL 512GB


In [16]:
import re

def normalize_text(text):
    if pd.isna(text):
        return None

    text = str(text).upper().strip()
    text = text.replace("_", " ")

    # Reemplazos seguros por palabra completa
    replacements = {
        r"\bSAMSNG\b": "SAMSUNG",
        r"\bAPLE\b": "APPLE",
        r"\bTV\b": "TELEVISOR",
        r"\bPANT\b": "PANTALLA",
        r"\bREFRI\b": "REFRIGERADOR",
        r"\bLICUAD\b": "LICUADORA",
    }

    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text)

    # Correcciones controladas de caracteres
    text = text.replace("MONIT0R", "MONITOR")
    text = text.replace("T3LEVISOR", "TELEVISOR")
    text = text.replace("APPL3", "APPLE")

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [17]:
products_sap["desc_norm"] = products_sap["description_sap"].apply(normalize_text)
products_as400["desc_norm"] = products_as400["description_as400"].apply(normalize_text)

In [18]:
products_as400[["description_as400", "desc_norm"]].head(15)

,description_as400,desc_norm
0,PANTALLA S,PANTALLA S
1,"BOCINA TCL 32""","BOCINA TCL 32"""
2,MONITOR JBL 256GB,MONITOR JBL 256GB
3,"CAMARA TCL 75""","CAMARA TCL 75"""
4,"CAMARA TCL 65""","CAMARA TCL 65"""
5,TABLET JBL 128GB,TABLET JBL 128GB
6,"MONITOR TCL 65""","MONITOR TCL 65"""
7,"TELEVISOR LG 32""","TELEVISOR LG 32"""
8,"TABLET LG 75""","TABLET LG 75"""
9,MONITOR TCL 512GB,MONITOR TCL 512GB


In [19]:
top_sap_by_price = (
    products_sap
    .sort_values("price_sap", ascending=False)
    .head(20)
    [["sku_sap", "description_sap", "category_sap", "price_sap"]]
)

top_sap_by_price

,sku_sap,description_sap,category_sap,price_sap
828,SAP-0000829,MONITOR HISENSE 512GB,ELECTRONICA,34998.49
53,SAP-0000054,"TELEVISOR JBL 32""",ELECTRONICA,34996.96
5686,SAP-0005687,LAPTOP LOGITECH 256GB,COMPUTO,34987.12
6034,SAP-0006035,DISCO DURO LOGITECH 512GB,COMPUTO,34984.72
5836,SAP-0005837,DISCO DURO ASUS 128GB,COMPUTO,34977.93
1944,SAP-0001945,CONSOLA SAMSUNG 512GB,ELECTRONICA,34972.71
2076,SAP-0002077,"CONSOLA HISENSE 55""",ELECTRONICA,34969.07
5861,SAP-0005862,"WEBCAM HP 65""",COMPUTO,34968.49
5670,SAP-0005671,WEBCAM ACER 128GB,COMPUTO,34956.41
1626,SAP-0001627,TELEVISOR SONY 256GB,ELECTRONICA,34955.12


In [20]:
ghost_sap_refs = apple_ghosts["sku_sap_reference"].dropna().tolist()

sap_ghost_details = products_sap[
    products_sap["sku_sap"].isin(ghost_sap_refs)
][["sku_sap", "description_sap", "category_sap", "brand", "price_sap"]]

sap_ghost_details.sort_values("price_sap", ascending=False).head(20)

,sku_sap,description_sap,category_sap,brand,price_sap
2114,SAP-0002115,FUNDA APPLE 128GB,TELEFONIA,APPLE,28059.41
2223,SAP-0002224,"IPHONE APPLE 65""",TELEFONIA,APPLE,28036.94
2145,SAP-0002146,"AIRPODS APPLE 55""",TELEFONIA,APPLE,27240.79
2199,SAP-0002200,"AIRPODS APPLE 75""",TELEFONIA,APPLE,26969.63
2119,SAP-0002120,"FUNDA APPLE 55""",TELEFONIA,APPLE,26302.09
2225,SAP-0002226,"GALAXY BUDS APPLE 65""",TELEFONIA,APPLE,25457.07
2150,SAP-0002151,FUNDA APPLE 128GB,TELEFONIA,APPLE,24974.43
2130,SAP-0002131,"CARGADOR APPLE 43""",TELEFONIA,APPLE,23478.72
2224,SAP-0002225,"GALAXY BUDS APPLE 55""",TELEFONIA,APPLE,23059.68
2233,SAP-0002234,CARGADOR APPLE 256GB,TELEFONIA,APPLE,20342.50


In [21]:
import sys
sys.executable

'/Users/evelyntorres/nova-retail-loss-prevention/venv/bin/python'

In [22]:
from fuzzywuzzy import process, fuzz
print("fuzzywuzzy cargado correctamente")

fuzzywuzzy cargado correctamente


In [23]:
sap_sample = products_sap.copy()
as400_sample = products_as400[products_as400["is_ghost"] == False].copy()

sap_sample = sap_sample.sort_values("price_sap", ascending=False).head(100)
as400_choices = as400_sample["desc_norm"].dropna().unique().tolist()

print("SAP sample:", sap_sample.shape)
print("AS400 choices:", len(as400_choices))

SAP sample: (100, 7)
AS400 choices: 4511


In [24]:
sample_matches = []

for _, row in sap_sample.iterrows():
    desc = row["desc_norm"]

    if pd.isna(desc):
        continue

    best_match = process.extractOne(
        desc,
        as400_choices,
        scorer=fuzz.token_sort_ratio
    )

    if best_match:
        match_desc, score = best_match
        sample_matches.append({
            "sku_sap": row["sku_sap"],
            "description_sap": row["description_sap"],
            "price_sap": row["price_sap"],
            "matched_desc_as400": match_desc,
            "fuzzy_score": score
        })

sample_matches_df = pd.DataFrame(sample_matches)
sample_matches_df.sort_values("fuzzy_score", ascending=False).head(20)

,sku_sap,description_sap,price_sap,matched_desc_as400,fuzzy_score
0,SAP-0000829,MONITOR HISENSE 512GB,34998.49,MONITOR HISENSE 512GB,100
63,SAP-0000199,"BOCINA SAMSUNG 32""",34290.19,"BOCINA SAMSUNG 32""",100
73,SAP-0001526,BOCINA JBL 256GB,34175.31,BOCINA JBL 256GB,100
72,SAP-0000568,BOCINA JBL 128GB,34193.34,BOCINA JBL 128GB,100
71,SAP-0006249,DESKTOP ACER 128GB,34227.34,DESKTOP ACER 128GB,100
70,SAP-0001922,"BOCINA HISENSE 43""",34231.78,"BOCINA HISENSE 43""",100
69,SAP-0005636,"DISCO DURO LOGITECH 75""",34234.60,"DISCO DURO LOGITECH 75""",100
68,SAP-0001318,"MONITOR TCL 55""",34248.66,"MONITOR TCL 55""",100
67,SAP-0001432,PANTALLA SONY 128GB,34249.25,PANTALLA SONY 128GB,100
66,SAP-0005996,"USB LOGITECH 55""",34253.55,"USB LOGITECH 55""",100


In [25]:
as400_lookup = products_as400[[
    "desc_norm", "price_as400", "sku_as400", "category_as400", "sku_sap_reference"
]].copy()

sample_matches_enriched = sample_matches_df.merge(
    as400_lookup,
    left_on="matched_desc_as400",
    right_on="desc_norm",
    how="left"
)

sample_matches_enriched[[
    "sku_sap",
    "description_sap",
    "price_sap",
    "matched_desc_as400",
    "price_as400",
    "fuzzy_score"
]].head(20)

,sku_sap,description_sap,price_sap,matched_desc_as400,price_as400,fuzzy_score
0,SAP-0000829,MONITOR HISENSE 512GB,34998.49,MONITOR HISENSE 512GB,14243.38,100
1,SAP-0000829,MONITOR HISENSE 512GB,34998.49,MONITOR HISENSE 512GB,20380.51,100
2,SAP-0000829,MONITOR HISENSE 512GB,34998.49,MONITOR HISENSE 512GB,5726.94,100
3,SAP-0000829,MONITOR HISENSE 512GB,34998.49,MONITOR HISENSE 512GB,362.41,100
4,SAP-0000829,MONITOR HISENSE 512GB,34998.49,MONITOR HISENSE 512GB,NaN,100
5,SAP-0000829,MONITOR HISENSE 512GB,34998.49,MONITOR HISENSE 512GB,32508.59,100
6,SAP-0000054,"TELEVISOR JBL 32""",34996.96,"TELEVISOR JBL 32""",34996.96,100
7,SAP-0000054,"TELEVISOR JBL 32""",34996.96,"TELEVISOR JBL 32""",17645.21,100
8,SAP-0000054,"TELEVISOR JBL 32""",34996.96,"TELEVISOR JBL 32""",NaN,100
9,SAP-0000054,"TELEVISOR JBL 32""",34996.96,"TELEVISOR JBL 32""",610.96,100


In [26]:
sample_matches_enriched["price_diff_abs"] = (
    sample_matches_enriched["price_sap"] - sample_matches_enriched["price_as400"]
).abs()

sample_matches_enriched["price_diff_pct"] = (
    sample_matches_enriched["price_diff_abs"] / sample_matches_enriched["price_sap"]
) * 100

sample_matches_enriched[[
    "sku_sap",
    "description_sap",
    "price_sap",
    "matched_desc_as400",
    "price_as400",
    "price_diff_pct",
    "fuzzy_score"
]].sort_values("price_diff_pct", ascending=False).head(20)

,sku_sap,description_sap,price_sap,matched_desc_as400,price_as400,price_diff_pct,fuzzy_score
478,SAP-0000077,CONSOLA JBL 512GB,33978.82,CONSOLA JBL 512GB,137.71,99.594718,100
426,SAP-0002098,"CAMARA HISENSE 65""",34062.82,"CAMARA HISENSE 65""",143.40,99.579013,100
268,SAP-0000781,PANTALLA TCL 128GB,34386.72,PANTALLA TCL 128GB,244.71,99.288359,100
36,SAP-0005671,WEBCAM ACER 128GB,34956.41,WEBCAM ACER 128GB,322.54,99.077308,100
3,SAP-0000829,MONITOR HISENSE 512GB,34998.49,MONITOR HISENSE 512GB,362.41,98.964498,100
272,SAP-0000781,PANTALLA TCL 128GB,34386.72,PANTALLA TCL 128GB,358.66,98.956981,100
33,SAP-0002077,"CONSOLA HISENSE 55""",34969.07,"CONSOLA HISENSE 55""",403.33,98.846609,100
431,SAP-0001078,"AUDIFONOS SONY 32""",34047.64,"AUDIFONOS SONY 32""",408.85,98.799183,100
61,SAP-0006150,"WEBCAM ACER 55""",34945.41,"WEBCAM ACER 55""",433.59,98.759236,100
133,SAP-0000727,"AUDIFONOS LG 32""",34734.39,"AUDIFONOS LG 32""",485.97,98.600897,100


In [27]:
def classify_match(row):
    if pd.isna(row["price_as400"]):
        return "AMBIGUO_PRECIO_NULO"
    elif row["fuzzy_score"] >= 90 and row["price_diff_pct"] <= 10:
        return "MATCH_CONFIABLE"
    elif row["fuzzy_score"] >= 90 and row["price_diff_pct"] > 10:
        return "AMBIGUO_PRECIO_INCONSISTENTE"
    else:
        return "MATCH_DEBIL"

sample_matches_enriched["match_status"] = sample_matches_enriched.apply(classify_match, axis=1)

sample_matches_enriched["match_status"].value_counts()

match_status
AMBIGUO_PRECIO_INCONSISTENTE    338
MATCH_CONFIABLE                 125
AMBIGUO_PRECIO_NULO              21
Name: count, dtype: int64

In [28]:
sample_matches_enriched[[
    "sku_sap",
    "description_sap",
    "matched_desc_as400",
    "price_sap",
    "price_as400",
    "price_diff_pct",
    "fuzzy_score",
    "match_status"
]].sort_values(["match_status", "price_diff_pct"], ascending=[True, False]).head(30)

,sku_sap,description_sap,matched_desc_as400,price_sap,price_as400,price_diff_pct,fuzzy_score,match_status
478,SAP-0000077,CONSOLA JBL 512GB,CONSOLA JBL 512GB,33978.82,137.71,99.594718,100,AMBIGUO_PRECIO_INCONSISTENTE
426,SAP-0002098,"CAMARA HISENSE 65""","CAMARA HISENSE 65""",34062.82,143.40,99.579013,100,AMBIGUO_PRECIO_INCONSISTENTE
268,SAP-0000781,PANTALLA TCL 128GB,PANTALLA TCL 128GB,34386.72,244.71,99.288359,100,AMBIGUO_PRECIO_INCONSISTENTE
36,SAP-0005671,WEBCAM ACER 128GB,WEBCAM ACER 128GB,34956.41,322.54,99.077308,100,AMBIGUO_PRECIO_INCONSISTENTE
3,SAP-0000829,MONITOR HISENSE 512GB,MONITOR HISENSE 512GB,34998.49,362.41,98.964498,100,AMBIGUO_PRECIO_INCONSISTENTE
...,...,...,...,...,...,...,...,...
471,SAP-0000971,"CAMARA SONY 43""","CAMARA SONY 43""",33992.51,1309.70,96.147092,100,AMBIGUO_PRECIO_INCONSISTENTE
368,SAP-0001162,"PANTALLA TCL 43""","PANTALLA TCL 43""",34149.51,1317.27,96.142639,100,AMBIGUO_PRECIO_INCONSISTENTE
108,SAP-0005219,LAPTOP ACER 256GB,LAPTOP ACER 256GB,34797.68,1380.57,96.032580,100,AMBIGUO_PRECIO_INCONSISTENTE
273,SAP-0000781,PANTALLA TCL 128GB,PANTALLA TCL 128GB,34386.72,1414.65,95.886057,100,AMBIGUO_PRECIO_INCONSISTENTE


In [29]:
products_as400["price_inferred_currency"] = np.where(
    products_as400["price_as400"].isna(),
    "UNKNOWN",
    np.where(products_as400["price_as400"] < 1000, "POTENTIAL_USD", "MXN")
)

products_as400["price_inferred_currency"].value_counts()

price_inferred_currency
POTENTIAL_USD    7149
MXN              6262
UNKNOWN           589
Name: count, dtype: int64

In [30]:
products_as400[[
    "description_as400",
    "category_as400",
    "price_as400",
    "price_inferred_currency"
]].sample(20, random_state=42)

,description_as400,category_as400,price_as400,price_inferred_currency
2900,"CARGADOR SAMSUNG 43""",CELULARES,1276.41,MXN
3143,"AIRPODS XIAOMI 55""",TELEFONIA,29436.16,MXN
8231,AGUA COCA-COLA 2L,NaN,322.00,POTENTIAL_USD
3855,ASPIRADORA OSTER 250G,ELECTRODOMESTICOS,1229.78,MXN
8045,JUGO MASECA 5KG,ALIMENTOS,70.10,POTENTIAL_USD
7653,AGUA LALA 5KG,ALIMENTOS,445.67,POTENTIAL_USD
8610,HUEVO COCA-COLA 1KG,ALIMENTOS,189.29,POTENTIAL_USD
7310,LECHE COCA-COLA 500ML,ALIMENTOS,425.70,POTENTIAL_USD
10175,CLORO FABULOSO 250G,LIMPIEZA,117.73,POTENTIAL_USD
5538,IMPRESORALOGITECH256GB,COMPUTO,20494.97,MXN


In [31]:
high_value_categories = [
    "ELECTRONICA", "TELEFONIA", "COMPUTO", "ELECTRODOMESTICOS"
]

def infer_currency(row):
    price = row["price_as400"]
    category = row["category_as400"]

    if pd.isna(price):
        return "UNKNOWN"

    if category in high_value_categories or category in ["ELECTRON", "ELECTR", "PANTALLAS", "AUDIO Y VIDEO", "TEL", "CELULARES", "MOVILES"]:
        if price < 1000:
            return "POTENTIAL_USD"
        else:
            return "MXN"

    return "MXN"

products_as400["price_inferred_currency_v2"] = products_as400.apply(infer_currency, axis=1)

products_as400["price_inferred_currency_v2"].value_counts()

price_inferred_currency_v2
MXN              13221
UNKNOWN            589
POTENTIAL_USD      190
Name: count, dtype: int64

In [32]:
products_as400[[
    "description_as400",
    "category_as400",
    "price_as400",
    "price_inferred_currency_v2"
]].sample(25, random_state=7)

,description_as400,category_as400,price_as400,price_inferred_currency_v2
3791,HORNO HAMILTON 12 PACK,ELECTRODOMESTICOS,5099.29,MXN
1682,"CAMARA HISENSE 75""",ELECTRONICA,19292.59,MXN
7267,AGUA LALA 100G,ABARROTES,188.70,MXN
8836,AGUA COCA-COLA 5KG,ALIMENTOS,22.61,MXN
1616,"MONITOR SONY 75""",ELECTRONICA,26765.72,MXN
...,...,...,...,...
3541,MICROONDASLG2L,ELECTRODOMESTICOS,2617.63,MXN
6968,LECHE COCA-COLA 1L,NaN,202.62,MXN
9631,SERVILLETAS PETALO 6 PACK,LIMP,250.26,MXN
4991,VENTILADOR MABE 250G,ELECTRODOMESTICOS,22613.07,MXN


In [34]:
EXCHANGE_RATE = 17.3

products_as400["price_as400_adjusted"] = np.where(
    products_as400["price_inferred_currency_v2"] == "POTENTIAL_USD",
    products_as400["price_as400"] * EXCHANGE_RATE,
    products_as400["price_as400"]
)

products_as400[[
    "description_as400",
    "category_as400",
    "price_as400",
    "price_inferred_currency_v2",
    "price_as400_adjusted"
]].sample(20, random_state=11)

,description_as400,category_as400,price_as400,price_inferred_currency_v2,price_as400_adjusted
1887,CONSOLA SAMSUNG 128GB,ELECTRONICA,NaN,UNKNOWN,NaN
10445,DESINFECTANTE DOWNY 100G,ASEO,6.33,MXN,6.33
1075,BOCINA HISENSE 128GB,NaN,7907.18,MXN,7907.18
9927,PAPEL HIGIENICO ROMA 5KG,LIMPIEZA,242.53,MXN,242.53
12653,VODKA JOHNNIE WALKER 250G,LICORES,4267.70,MXN,4267.70
5458,DESKTOP HP 128GB,COMPUTO,12787.71,MXN,12787.71
799,"CONSOLA JBL 75""",ELECTRONICA,2211.87,MXN,2211.87
3444,"GALAXY BUDS HUAWEI 65""",TELEFONIA,15256.67,MXN,15256.67
8379,HUEVO LALA 6 PACK,NaN,56.49,MXN,56.49
2899,"IPHONE APPLE 75""",NaN,12675.46,MXN,12675.46


In [35]:
as400_lookup_v2 = products_as400[[
    "desc_norm",
    "price_as400_adjusted",
    "sku_as400",
    "category_as400",
    "sku_sap_reference",
    "price_inferred_currency_v2"
]].copy()

sample_matches_enriched_v2 = sample_matches_df.merge(
    as400_lookup_v2,
    left_on="matched_desc_as400",
    right_on="desc_norm",
    how="left"
)

sample_matches_enriched_v2["price_diff_abs"] = (
    sample_matches_enriched_v2["price_sap"] - sample_matches_enriched_v2["price_as400_adjusted"]
).abs()

sample_matches_enriched_v2["price_diff_pct"] = (
    sample_matches_enriched_v2["price_diff_abs"] / sample_matches_enriched_v2["price_sap"]
) * 100

sample_matches_enriched_v2[[
    "sku_sap",
    "description_sap",
    "price_sap",
    "matched_desc_as400",
    "price_as400_adjusted",
    "price_diff_pct",
    "price_inferred_currency_v2",
    "fuzzy_score"
]].sort_values("price_diff_pct").head(25)

,sku_sap,description_sap,price_sap,matched_desc_as400,price_as400_adjusted,price_diff_pct,price_inferred_currency_v2,fuzzy_score
406,SAP-0002023,"BOCINA JBL 55""",34097.80,"BOCINA JBL 55""",34097.80,0.0,MXN,100
296,SAP-0000199,"BOCINA SAMSUNG 32""",34290.19,"BOCINA SAMSUNG 32""",34290.19,0.0,MXN,100
438,SAP-0000466,MONITOR SAMSUNG 256GB,34044.33,MONITOR SAMSUNG 256GB,34044.33,0.0,MXN,100
200,SAP-0001120,AUDIFONOS TCL 512GB,34659.45,AUDIFONOS TCL 512GB,34659.45,0.0,MXN,100
441,SAP-0001140,"PANTALLA SONY 43""",34024.43,"PANTALLA SONY 43""",34024.43,0.0,MXN,100
...,...,...,...,...,...,...,...,...
82,SAP-0000701,TABLET JBL 256GB,34884.89,TABLET JBL 256GB,34884.89,0.0,MXN,100
397,SAP-0001638,TABLET TCL 128GB,34132.20,TABLET TCL 128GB,34132.20,0.0,MXN,100
246,SAP-0005832,"MOUSE LOGITECH 55""",34536.56,"MOUSE LOGITECH 55""",34536.56,0.0,MXN,100
386,SAP-0001717,PANTALLA SAMSUNG 256GB,34139.96,PANTALLA SAMSUNG 256GB,34139.96,0.0,MXN,100


In [36]:
def classify_match_v2(row):
    if pd.isna(row["price_as400_adjusted"]):
        return "AMBIGUO_PRECIO_NULO"
    elif row["fuzzy_score"] >= 90 and row["price_diff_pct"] <= 10:
        return "MATCH_CONFIABLE"
    elif row["fuzzy_score"] >= 90 and row["price_diff_pct"] > 10:
        return "AMBIGUO_PRECIO"
    else:
        return "MATCH_DEBIL"

sample_matches_enriched_v2["match_status_v2"] = sample_matches_enriched_v2.apply(classify_match_v2, axis=1)

sample_matches_enriched_v2["match_status_v2"].value_counts()

match_status_v2
AMBIGUO_PRECIO         338
MATCH_CONFIABLE        125
AMBIGUO_PRECIO_NULO     21
Name: count, dtype: int64

In [37]:
sample_matches_enriched_v2[[
    "sku_sap",
    "description_sap",
    "matched_desc_as400",
    "price_sap",
    "price_as400_adjusted",
    "price_diff_pct",
    "fuzzy_score",
    "match_status_v2"
]].sort_values(["match_status_v2", "price_diff_pct"], ascending=[True, True]).head(30)

,sku_sap,description_sap,matched_desc_as400,price_sap,price_as400_adjusted,price_diff_pct,fuzzy_score,match_status_v2
424,SAP-0002098,"CAMARA HISENSE 65""","CAMARA HISENSE 65""",34062.82,30644.99,10.033902,100,AMBIGUO_PRECIO
123,SAP-0006241,"WEBCAM DELL 75""","WEBCAM DELL 75""",34737.13,31235.38,10.080712,100,AMBIGUO_PRECIO
467,SAP-0000848,CONSOLA LG 128GB,CONSOLA LG 128GB,34001.04,30470.22,10.384447,100,AMBIGUO_PRECIO
240,SAP-0001216,"TABLET HISENSE 55""","TABLET HISENSE 55""",34544.82,30848.57,10.699868,100,AMBIGUO_PRECIO
164,SAP-0000234,CONSOLA SAMSUNG 512GB,CONSOLA SAMSUNG 512GB,34693.12,30942.61,10.810530,100,AMBIGUO_PRECIO
...,...,...,...,...,...,...,...,...
384,SAP-0001717,PANTALLA SAMSUNG 256GB,PANTALLA SAMSUNG 256GB,34139.96,28122.56,17.625680,100,AMBIGUO_PRECIO
430,SAP-0001078,"AUDIFONOS SONY 32""","AUDIFONOS SONY 32""",34047.64,27994.03,17.779823,100,AMBIGUO_PRECIO
440,SAP-0001140,"PANTALLA SONY 43""","PANTALLA SONY 43""",34024.43,27897.08,18.008678,100,AMBIGUO_PRECIO
379,SAP-0001589,"TELEVISOR JBL 43""","TELEVISOR JBL 43""",34148.64,27969.75,18.094103,100,AMBIGUO_PRECIO


In [38]:
sample_matches_enriched_v2["category_sap"] = sample_matches_enriched_v2["sku_sap"].map(
    products_sap.set_index("sku_sap")["category_sap"]
)

ambiguos = sample_matches_enriched_v2[
    sample_matches_enriched_v2["match_status_v2"] == "AMBIGUO_PRECIO"
]

ambiguos["category_sap"].value_counts()

category_sap
ELECTRONICA    270
COMPUTO         68
Name: count, dtype: int64

In [39]:
ambiguos[[
    "sku_sap",
    "description_sap",
    "category_sap",
    "price_sap",
    "price_as400_adjusted",
    "price_diff_pct",
    "fuzzy_score"
]].sort_values("price_diff_pct", ascending=False).head(25)

,sku_sap,description_sap,category_sap,price_sap,price_as400_adjusted,price_diff_pct,fuzzy_score
3,SAP-0000829,MONITOR HISENSE 512GB,ELECTRONICA,34998.49,362.410,98.964498,100
272,SAP-0000781,PANTALLA TCL 128GB,ELECTRONICA,34386.72,358.660,98.956981,100
33,SAP-0002077,"CONSOLA HISENSE 55""",ELECTRONICA,34969.07,403.330,98.846609,100
365,SAP-0001162,"PANTALLA TCL 43""",ELECTRONICA,34149.51,551.290,98.385658,100
375,SAP-0000318,"BOCINA LG 75""",ELECTRONICA,34149.24,747.050,97.812396,100
...,...,...,...,...,...,...,...
290,SAP-0005613,WEBCAM LOGITECH 256GB,COMPUTO,34303.42,1982.860,94.219643,100
68,SAP-0001396,"PANTALLA LG 43""",ELECTRONICA,34941.45,2394.580,93.146879,100
399,SAP-0001638,TABLET TCL 128GB,ELECTRONICA,34132.20,2387.540,93.005022,100
478,SAP-0000077,CONSOLA JBL 512GB,ELECTRONICA,33978.82,2382.383,92.988624,100


In [40]:
products_sap["priority_flag"] = np.where(
    products_sap["price_sap"] >= products_sap["price_sap"].quantile(0.95),
    "TOP_5_VALUE",
    "STANDARD"
)

priority_skus = products_sap[products_sap["priority_flag"] == "TOP_5_VALUE"].copy()

print("SKUs prioritarios:", len(priority_skus))
priority_skus[["sku_sap", "description_sap", "category_sap", "price_sap"]].head(20)

SKUs prioritarios: 700


,sku_sap,description_sap,category_sap,price_sap
3,SAP-0000004,"CAMARA TCL 75""",ELECTRONICA,28635.23
7,SAP-0000008,"TELEVISOR LG 32""",ELECTRONICA,28566.51
10,SAP-0000011,"TABLET TCL 75""",ELECTRONICA,34045.59
11,SAP-0000012,"MONITOR HISENSE 32""",ELECTRONICA,33475.93
14,SAP-0000015,MONITOR LG 512GB,ELECTRONICA,33246.01
25,SAP-0000026,CONSOLA JBL 128GB,ELECTRONICA,32845.31
28,SAP-0000029,"TELEVISOR JBL 75""",ELECTRONICA,32747.95
31,SAP-0000032,"MONITOR TCL 32""",ELECTRONICA,29541.64
35,SAP-0000036,"TABLET SONY 75""",ELECTRONICA,29077.70
44,SAP-0000045,"TABLET SAMSUNG 65""",ELECTRONICA,32276.51


In [41]:
products_sap["priority_flag"] = np.where(
    products_sap["price_sap"] >= products_sap["price_sap"].quantile(0.95),
    "TOP_5_VALUE",
    "STANDARD"
)

priority_skus = products_sap[products_sap["priority_flag"] == "TOP_5_VALUE"].copy()

print("SKUs prioritarios:", len(priority_skus))
priority_skus[["sku_sap", "description_sap", "category_sap", "price_sap"]].head(20)

SKUs prioritarios: 700


,sku_sap,description_sap,category_sap,price_sap
3,SAP-0000004,"CAMARA TCL 75""",ELECTRONICA,28635.23
7,SAP-0000008,"TELEVISOR LG 32""",ELECTRONICA,28566.51
10,SAP-0000011,"TABLET TCL 75""",ELECTRONICA,34045.59
11,SAP-0000012,"MONITOR HISENSE 32""",ELECTRONICA,33475.93
14,SAP-0000015,MONITOR LG 512GB,ELECTRONICA,33246.01
25,SAP-0000026,CONSOLA JBL 128GB,ELECTRONICA,32845.31
28,SAP-0000029,"TELEVISOR JBL 75""",ELECTRONICA,32747.95
31,SAP-0000032,"MONITOR TCL 32""",ELECTRONICA,29541.64
35,SAP-0000036,"TABLET SONY 75""",ELECTRONICA,29077.70
44,SAP-0000045,"TABLET SAMSUNG 65""",ELECTRONICA,32276.51


In [42]:
resumen_reconciliacion = pd.DataFrame({
    "indicador": [
        "SKUs SAP totales",
        "SKUs AS400 totales",
        "Apple Ghosts",
        "SKUs prioritarios (Top 5% valor)",
        "Matches piloto evaluados",
        "Matches confiables (piloto)",
        "Matches ambiguos (piloto)",
        "Precios AS400 nulos"
    ],
    "valor": [
        len(products_sap),
        len(products_as400),
        int(products_as400["is_ghost"].sum()),
        int((products_sap["priority_flag"] == "TOP_5_VALUE").sum()),
        len(sample_matches_enriched_v2),
        int((sample_matches_enriched_v2["match_status_v2"] == "MATCH_CONFIABLE").sum()),
        int((sample_matches_enriched_v2["match_status_v2"] == "AMBIGUO_PRECIO").sum()),
        int(products_as400["price_as400"].isna().sum())
    ]
})

resumen_reconciliacion

,indicador,valor
0,SKUs SAP totales,14000
1,SKUs AS400 totales,14000
2,Apple Ghosts,22
3,SKUs prioritarios (Top 5% valor),700
4,Matches piloto evaluados,484
5,Matches confiables (piloto),125
6,Matches ambiguos (piloto),338
7,Precios AS400 nulos,589


## Conclusiones de la fase de reconciliación

### Hallazgos principales
- El catálogo AS400 presenta problemas estructurales de trazabilidad, especialmente por categorías vacías, precios nulos y productos invisibles.
- Los **Apple Ghosts** representan un riesgo crítico porque existen en SAP pero no en AS400, impidiendo su seguimiento completo en el entorno legacy.
- La normalización textual permitió construir una primera capa funcional de matching entre catálogos.
- El matching difuso por descripción, sin validación financiera, genera equivalencias semánticamente plausibles pero financieramente inconsistentes.
- La inferencia de moneda mejoró materialmente la comparabilidad de precios en productos de alto valor.
- La reconciliación total del catálogo no es operativamente eficiente en esta fase; el enfoque correcto es una **reconciliación selectiva por valor**.

### Decisión metodológica
La siguiente etapa del proyecto debe priorizar los SKUs de mayor valor económico, en lugar de intentar una limpieza horizontal de los 14,000 productos. Esta decisión maximiza la recuperación potencial y reduce el tiempo de diagnóstico.

### Riesgo residual
Persisten ambigüedades en una parte relevante del catálogo legacy, especialmente en productos de alto ticket con discrepancias de valuación. Estos casos requieren tratamiento controlado antes de escalar a reconciliación masiva.

## Próximo paso recomendado

Con la reconciliación inicial de SKUs establecida, la siguiente fase del análisis debe enfocarse en:

1. **Cruzar despachos del CEDIS contra recepciones en tienda**
2. **Normalizar fechas y unidades de recepción**
3. **Medir varianzas selectivas por tienda, categoría y horario**
4. **Identificar tiendas con patrones incompatibles con error operativo aleatorio**

El objetivo ya no es solo limpiar datos, sino comenzar a separar:
- ruido administrativo
- falla operativa
- y pérdida potencialmente intencional
